# Binary Classification EDA 
This notebook performs a complete Step 1 EDA for a binary classification dataset.

- Dataset source: `../data/raw/diabetes.csv`
- Working DataFrame: `df`
- Target column expected: `target` (auto-renamed from `Outcome` if needed)

In [4]:
# Imports and basic setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# Load dataset
df = pd.read_csv("../data/raw/diabetes.csv")

# Ensure target column name is `target`
if "target" not in df.columns and "Outcome" in df.columns:
    df = df.rename(columns={"Outcome": "target"})

print("Dataset loaded successfully.")
print(f"Shape: {df.shape}")
print(f"Target column present: {'target' in df.columns}")

Dataset loaded successfully.
Shape: (768, 9)
Target column present: False


In [5]:
# ── 1. DATA OVERVIEW ──
print("=" * 70)
print("1) DATA OVERVIEW")
print("=" * 70)

# Shape
print(f"\nShape: {df.shape}")

# Column names
print("\nColumns:")
print(df.columns.tolist())

# Data types
print("\nData types:")
print(df.dtypes)

# First 5 rows
print("\nFirst 5 rows:")
display(df.head())

# Missing values (count + percentage)
missing_count = df.isna().sum()
missing_pct = (missing_count / len(df)) * 100
missing_summary = pd.DataFrame({
    "missing_count": missing_count,
    "missing_percentage": missing_pct
}).sort_values("missing_percentage", ascending=False)

print("\nMissing values summary:")
display(missing_summary)

1) DATA OVERVIEW

Shape: (768, 9)

Columns:
['pregnancies', 'glucose', 'bloodpressure', 'skinthickness', 'insulin', 'bmi', 'diabetes_pedigree_function', 'age', 'outcome']

Data types:
pregnancies                     int64
glucose                         int64
bloodpressure                   int64
skinthickness                   int64
insulin                         int64
bmi                           float64
diabetes_pedigree_function    float64
age                             int64
outcome                         int64
dtype: object

First 5 rows:


,pregnancies,glucose,bloodpressure,skinthickness,insulin,bmi,diabetes_pedigree_function,age,outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1



Missing values summary:


,missing_count,missing_percentage
pregnancies,0,0.0
glucose,0,0.0
bloodpressure,0,0.0
skinthickness,0,0.0
insulin,0,0.0
bmi,0,0.0
diabetes_pedigree_function,0,0.0
age,0,0.0
outcome,0,0.0


In [6]:
# ── 2. CLASS BALANCE ──
print("=" * 70)
print("2) CLASS BALANCE")
print("=" * 70)

if "target" not in df.columns:
    raise ValueError("Column 'target' not found. Please rename your target column to 'target'.")

class_counts = df["target"].value_counts().sort_index()
count_0 = int(class_counts.get(0, 0))
count_1 = int(class_counts.get(1, 0))

print(f"Count of class 0: {count_0}")
print(f"Count of class 1: {count_1}")

# Bar chart for class distribution
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(x=class_counts.index.astype(str), y=class_counts.values, palette="Set2", ax=ax)
ax.set_title("Class Distribution (target)")
ax.set_xlabel("Class")
ax.set_ylabel("Count")

# Label counts on bars
for i, value in enumerate(class_counts.values):
    ax.text(i, value, f"{value}", ha="center", va="bottom", fontweight="bold")

plt.tight_layout()
plt.show()

2) CLASS BALANCE


ValueError: Column 'target' not found. Please rename your target column to 'target'.

In [ ]:
# ── 3. FEATURE DISTRIBUTIONS ──
print("=" * 70)
print("3) FEATURE DISTRIBUTIONS")
print("=" * 70)

# Separate numerical and categorical features
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()

# Exclude target from feature plotting if present
if "target" in numerical_cols:
    numerical_feature_cols = [c for c in numerical_cols if c != "target"]
else:
    numerical_feature_cols = numerical_cols

print(f"Numerical features ({len(numerical_feature_cols)}): {numerical_feature_cols}")
print(f"Categorical features ({len(categorical_cols)}): {categorical_cols}")

# Numerical feature histograms with KDE in a grid
if len(numerical_feature_cols) > 0:
    n = len(numerical_feature_cols)
    n_cols = 3
    n_rows = int(np.ceil(n / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
    axes = np.array(axes).reshape(-1)

    for i, col in enumerate(numerical_feature_cols):
        sns.histplot(df[col].dropna(), kde=True, ax=axes[i], color="steelblue")
        axes[i].set_title(f"Distribution of {col}")
        axes[i].set_xlabel(col)
        axes[i].set_ylabel("Frequency")

    # Remove empty subplots
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    plt.show()

# Categorical feature bar charts
if len(categorical_cols) > 0:
    for col in categorical_cols:
        value_counts = df[col].value_counts(dropna=False)

        fig, ax = plt.subplots(figsize=(9, 5))
        sns.barplot(x=value_counts.index.astype(str), y=value_counts.values, palette="viridis", ax=ax)
        ax.set_title(f"Value Counts - {col}")
        ax.set_xlabel(col)
        ax.set_ylabel("Count")
        ax.tick_params(axis="x", rotation=45)

        for idx, val in enumerate(value_counts.values):
            ax.text(idx, val, str(val), ha="center", va="bottom", fontsize=9)

        plt.tight_layout()
        plt.show()
else:
    print("No categorical features found.")

In [ ]:
# ── 4. CORRELATION ANALYSIS ──
print("=" * 70)
print("4) CORRELATION ANALYSIS")
print("=" * 70)

# Correlation matrix (numeric only)
numeric_df = df.select_dtypes(include=[np.number]).copy()

if numeric_df.shape[1] < 2:
    print("Not enough numeric columns to compute correlation matrix.")
else:
    corr_matrix = numeric_df.corr()

    # Heatmap
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", square=True)
    plt.title("Correlation Heatmap")
    plt.xlabel("Features")
    plt.ylabel("Features")
    plt.tight_layout()
    plt.show()

    # Top 10 features most correlated with target
    if "target" in corr_matrix.columns:
        target_corr = corr_matrix["target"].drop("target", errors="ignore").sort_values(ascending=False)

        print("Top 10 features most correlated with target (descending):")
        print(target_corr.head(10))
    else:
        print("Column 'target' is not numeric or not present in numeric columns.")